In [ ]:
#1 AMBIENTE TENSORFLOW/KERAS

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.metrics import mean_absolute_error, mean_squared_error

RANDOM_SEED = 42
MIN_OBSERVACIONES = 9
EPOCHS = 20
BATCH_SIZE = 1

FILAS = 271
COLUMNAS = 446

FECHA_INICIO = pd.Timestamp("2015-05-01")
FECHA_FIN = pd.Timestamp("2026-08-01")

FECHA_INICIO_PRONOSTICO = pd.Timestamp("2026-09-01")
FECHA_FIN_PRONOSTICO = pd.Timestamp("2027-08-01")

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print("TensorFlow:", tf.__version__)
print("Dispositivos:", tf.config.list_physical_devices())
print("MIN_OBSERVACIONES:", MIN_OBSERVACIONES)
print("EPOCHS:", EPOCHS)

In [ ]:
#2 AJUSTAR ÚNICAMENTE ESTA RUTA A LA UBICACIÓN REAL DEL NPZ

RUTA_DATOS = r"C:\RUTA\A\datos_lstm_actual.npz"

datos = np.load(RUTA_DATOS)

datos_validos = datos["datos_validos"].astype(np.float32)
indices_validos = datos["indices_validos"].astype(np.int32)

print("datos_validos:", datos_validos.shape)
print("indices_validos:", indices_validos.shape)

In [ ]:
#3 CALENDARIO HISTÓRICO

fechas = pd.date_range(FECHA_INICIO, FECHA_FIN, freq="MS")

if len(fechas) != datos_validos.shape[1]:
    raise ValueError(
        f"Hay {len(fechas)} fechas pero {datos_validos.shape[1]} temporalidades en datos_validos."
    )

print("Temporalidades:", len(fechas))
print("Periodo:", fechas[0], "→", fechas[-1])

In [ ]:
# SERIE HISTÓRICA (TIEMPO, FILAS, COLUMNAS)

serie_historica = np.zeros(
    (len(fechas), FILAS, COLUMNAS),
    dtype=np.float32
)

for t in range(len(fechas)):
    serie_historica[t].flat[indices_validos] = datos_validos[:, t]

print("Serie:", serie_historica.shape)
print("Rango:", float(serie_historica.min()), "→", float(serie_historica.max()))

In [ ]:
# ÍNDICES DE OBSERVACIONES POR MES

indices_por_mes = {
    mes: np.where(fechas.month == mes)[0]
    for mes in range(1, 13)
}

for mes in range(1, 13):
    print(f"Mes {mes:02d}: {len(indices_por_mes[mes])} observaciones")

In [ ]:
# CONSTRUIR EJEMPLOS HISTÓRICOS INTERANUALES

X_list = []
y_list = []
fechas_objetivo = []
longitudes = []

for i, fecha_objetivo in enumerate(fechas):
    anteriores = [
        j for j in indices_por_mes[fecha_objetivo.month]
        if j < i
    ]

    if len(anteriores) < MIN_OBSERVACIONES:
        continue

    X_list.append(serie_historica[anteriores][..., np.newaxis])
    y_list.append(serie_historica[i][..., np.newaxis])
    fechas_objetivo.append(fecha_objetivo)
    longitudes.append(len(anteriores))

fechas_objetivo = pd.DatetimeIndex(fechas_objetivo)

print("Ejemplos:", len(X_list))
print("Primer objetivo:", fechas_objetivo[0])
print("Último objetivo:", fechas_objetivo[-1])
print("Longitudes:", longitudes)

In [ ]:
# PADDING TEMPORAL PARA FORMAR EL TENSOR DE KERAS

MAX_TIMESTEPS = max(longitudes)

X = np.zeros(
    (len(X_list), MAX_TIMESTEPS, FILAS, COLUMNAS, 1),
    dtype=np.float32
)

for i, secuencia in enumerate(X_list):
    n = secuencia.shape[0]
    X[i, -n:] = secuencia

y = np.asarray(y_list, dtype=np.float32)

print("X:", X.shape)
print("y:", y.shape)
print("MAX_TIMESTEPS:", MAX_TIMESTEPS)

In [ ]:
# VALIDACIÓN: ABRIL, MAYO Y JUNIO DE 2026

mask_train = fechas_objetivo < pd.Timestamp("2026-04-01")
mask_val = (
    (fechas_objetivo >= pd.Timestamp("2026-04-01")) &
    (fechas_objetivo <= pd.Timestamp("2026-06-01"))
)

X_train = X[mask_train]
y_train = y[mask_train]
fechas_train = fechas_objetivo[mask_train]

X_val = X[mask_val]
y_val = y[mask_val]
fechas_val = fechas_objetivo[mask_val]

print("TRAIN:", X_train.shape, y_train.shape)
print("VALIDACIÓN:", X_val.shape, y_val.shape)
print("Fechas validación:", list(fechas_val))

In [ ]:
# ARQUITECTURA CONVLSTM INTERANUAL

modelo = Sequential([
    
    ConvLSTM2D(
        filters=16,
        kernel_size=(3, 3),
        padding="same",
        activation="tanh",
        return_sequences=True,
        input_shape=(MAX_TIMESTEPS, 271, 446, 1)
    ),

    BatchNormalization(),

    ConvLSTM2D(
        filters=8,
        kernel_size=(3, 3),
        padding="same",
        activation="tanh",
        return_sequences=False
    ),

    BatchNormalization(),

    Conv2D(
        filters=1,
        kernel_size=(3, 3),
        padding="same",
        activation="sigmoid"
    )
])

modelo.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mae",
    metrics=["mae"]
)

modelo.summary()

In [ ]:
# Entrenamiento MÁXIMO 20 ÉPOCAS

historial = modelo.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

In [ ]:
# CURVA DE ENTRENAMIENTO

plt.figure(figsize=(10, 5))
plt.plot(historial.history["loss"], label="Entrenamiento")
plt.plot(historial.history["val_loss"], label="Validación")
plt.xlabel("Época")
plt.ylabel("MAE")
plt.title("ConvLSTM interanual — pérdida")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# MÉTRICAS DE VALIDACIÓN

pred_val = modelo.predict(X_val, batch_size=BATCH_SIZE, verbose=1)

resultados = []

for i, fecha in enumerate(fechas_val):
    observado = y_val[i, ..., 0].ravel()
    predicho = pred_val[i, ..., 0].ravel()

    mae = mean_absolute_error(observado, predicho)
    rmse = np.sqrt(mean_squared_error(observado, predicho))
    corr = np.corrcoef(observado, predicho)[0, 1]

    resultados.append({
        "objetivo": fecha,
        "MAE": mae,
        "RMSE": rmse,
        "correlacion": corr
    })

df_validacion = pd.DataFrame(resultados)
display(df_validacion)

In [ ]:
#Creación FECHAS FUTURAS

fechas_pronostico = pd.date_range(
    FECHA_INICIO_PRONOSTICO,
    FECHA_FIN_PRONOSTICO,
    freq="MS"
)

print(list(fechas_pronostico))

In [ ]:
# ENTRADAS INTERANUALES DEL PRONÓSTICO

X_forecast = np.zeros(
    (len(fechas_pronostico), MAX_TIMESTEPS, FILAS, COLUMNAS, 1),
    dtype=np.float32
)

longitudes_forecast = []

for i, fecha in enumerate(fechas_pronostico):
    indices_mes = indices_por_mes[fecha.month]
    secuencia = serie_historica[indices_mes]
    n = len(secuencia)

    if n < MIN_OBSERVACIONES:
        raise ValueError(f"{fecha:%Y-%m}: {n} observaciones < {MIN_OBSERVACIONES}")
    if n > MAX_TIMESTEPS:
        raise ValueError(f"{fecha:%Y-%m}: {n} observaciones > {MAX_TIMESTEPS}")

    X_forecast[i, -n:, :, :, 0] = secuencia
    longitudes_forecast.append(n)

for fecha, n in zip(fechas_pronostico, longitudes_forecast):
    print(f"{fecha:%Y-%m}: {n} años")

In [ ]:
# GENERAR PRONÓSTICOS

predicciones = modelo.predict(
    X_forecast,
    batch_size=BATCH_SIZE,
    verbose=1
)[..., 0]

print("Forma:", predicciones.shape)
print("Rango:", predicciones.min(), "→", predicciones.max())

In [ ]:
# GUARDAR NPZ FINAL

CARPETA_SALIDA = r"C:\Users\57312\Documents\Proyectos Esri\Charla"

RUTA_SALIDA = os.path.join(
    CARPETA_SALIDA,
    "pronosticos_convlstm_interanual_2026_2027.npz"
)

os.makedirs(CARPETA_SALIDA, exist_ok=True)

np.savez_compressed(
    RUTA_SALIDA,
    predicciones=predicciones.astype(np.float32),
    fechas=np.array(fechas_pronostico.strftime("%Y-%m-%d"), dtype="U10")
)

print("Guardado:")
print(RUTA_SALIDA)